In [84]:
import os
import math
import pandas as pd
import numpy as np
from pathlib import Path
import xml.dom.minidom 
from datetime import datetime, timedelta
import xml.etree.ElementTree as Et

# paga o caminho para a raiz database
def get_xml_root():
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()
    return os.path.join(BASE_DIR, "..", "OhioT1DM")

# paga o caminho para os arquivos XML
def get_XMLs(root):
    p = Path(root)
    lista_XML = []

    for x in p.iterdir():
        if x.is_dir() :
            lista_XML.extend(get_XMLs(x))
        # elif "testing" in x.name: 
        #     continue
        # elif "training" in x.name: 
        #     continue
        elif x.suffix == ".xml": 
            lista_XML.append(x)

    return lista_XML

# estrai as informaçõens dos XMLs
def get_info(file_XML, dados: dict):
    domtree = xml.dom.minidom.parse(str(file_XML))

    patient = domtree.documentElement
    assert patient is not None

    id_patient = int(patient.getAttribute('id'))
    
    if id_patient not in dados.keys():
        dados[id_patient] = {}

    finger_stick = patient.getElementsByTagName('finger_stick')[0].getElementsByTagName('event')
    for event in finger_stick:
        ts =  binning(event.getAttribute('ts'))
        dados = new_entry(ts, dados, id_patient)
        dados[id_patient][ts]["metodo_medida"] = "finger_stick"
        dados[id_patient][ts]["glucose_level"] = int(event.getAttribute('value'))

    glucose_level = patient.getElementsByTagName('glucose_level')[0].getElementsByTagName('event')
    for event in glucose_level:
        ts =  binning(event.getAttribute('ts'))
        dados = new_entry(ts, dados, id_patient)
        dados[id_patient][ts]["metodo_medida"] = "CGM"
        dados[id_patient][ts]["glucose_level"] = int(event.getAttribute('value'))           

    bolus = patient.getElementsByTagName('bolus')[0].getElementsByTagName('event')
    for event in bolus:
        ts =  binning(event.getAttribute('ts_begin'))
        dados = new_entry(ts, dados, id_patient)
        dados[id_patient][ts]["bolus"] = event.getAttribute('dose')
        
    meal = patient.getElementsByTagName('meal')[0].getElementsByTagName('event')
    for event in meal:
        ts =  binning(event.getAttribute('ts'))
        dados = new_entry(ts, dados, id_patient)
        dados[id_patient][ts]["meal_type"] = event.getAttribute('type')
        dados[id_patient][ts]["meal_carbs"] = event.getAttribute('carbs')

    basal = patient.getElementsByTagName('basal')[0].getElementsByTagName('event')
    for event in basal:
        ts =  binning(event.getAttribute('ts'))

        for entry in dados[id_patient]:
            if entry >= ts:
                dados[id_patient][entry]["basal"] = float(event.getAttribute('value'))

    temp_basal = patient.getElementsByTagName('temp_basal')[0].getElementsByTagName('event')
    for event in temp_basal:
        ts_begin =  binning(event.getAttribute('ts_begin'))
        ts_end =  binning(event.getAttribute('ts_end'))

        for entry in dados[id_patient]:
            if entry >= ts_begin and entry <= ts_end:
                dados[id_patient][entry]["basal"] = float(event.getAttribute('value'))

    sleep = patient.getElementsByTagName('sleep')[0].getElementsByTagName('event')
    for event in sleep:
        ts_begin =  binning(event.getAttribute('ts_begin'))
        ts_end =  binning(event.getAttribute('ts_end'))

        for entry in dados[id_patient]:
            if entry >= ts_begin and entry <= ts_end:
                dados[id_patient][entry]["sleeping"] = True
                dados[id_patient][entry]["sleep_quality"] = event.getAttribute('quality')

    exercise = patient.getElementsByTagName('exercise')[0].getElementsByTagName('event')
    for event in exercise:
        ts_begin =  binning(event.getAttribute('ts'))
        ts_end = ts_begin + timedelta(minutes= int(event.getAttribute('duration')))

        for entry in dados[id_patient]:
            if entry >= ts and entry <= ts_end:
                dados[id_patient][entry]["doing_exercise"] = True
                dados[id_patient][entry]["exercise_intensity"] = event.getAttribute('intensity')

    
    return dados

# cria uma nova entrada caso ela não existir
def new_entry(ts, dados: dict, id_patient):
    if ts not in dados[id_patient].keys():
        dados[id_patient][ts] = {"id_patient": id_patient,
                                 "ts": ts,
                                 "metodo_medida": None, 
                                 "glucose_level": None,
                                 "basal": None,
                                 "bolus": None,
                                 "meal_type": None, 
                                 "meal_carbs": None,
                                 "sleeping": False,
                                 "sleep_quality": None,
                                 "exercise_intensity": None,
                                 "doing_exercise": False}
    return dados

# aredonda o horario para o 5 muinutos enterior 
def binning(ts): 
    data = datetime.strptime(ts, "%d-%m-%Y %H:%M:%S")
    i = 5
    mim = data.minute//i*i
    return data.replace(minute = mim, second = 0)
    
lista_XML = get_XMLs(get_xml_root())

para = 0
dados_individual = {}
dados_geral = []
for file in lista_XML:
    # if para > 0:
    #     break 
    dados_individual = get_info(file, dados_individual)
    # para +=1


for pacient in dados_individual:
    for ts in dados_individual[pacient]:
        dados_geral.append(dados_individual[pacient][ts])

df_paciente = pd.DataFrame(dados_geral)
# df_paciente = pd.DataFrame(dados_individual)
display(df_paciente)

,id_patient,ts,metodo_medida,glucose_level,basal,bolus,meal_type,meal_carbs,sleeping,sleep_quality,exercise_intensity,doing_exercise
0,559,2022-01-18 06:25:00,CGM,249.0,0.83,6.3,NaN,NaN,False,NaN,NaN,False
1,559,2022-01-18 16:55:00,CGM,150.0,0.83,NaN,NaN,NaN,False,NaN,NaN,False
2,559,2022-01-18 17:55:00,finger_stick,134.0,0.83,0.2,NaN,NaN,False,NaN,NaN,False
3,559,2022-01-19 04:30:00,CGM,96.0,0.83,NaN,NaN,NaN,False,NaN,NaN,False
4,559,2022-01-19 05:30:00,CGM,161.0,0.83,1.2,NaN,NaN,False,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...
167700,596,2027-05-24 10:15:00,NaN,NaN,0.60,NaN,Snack,15,False,NaN,NaN,False
167701,596,2027-05-24 12:10:00,NaN,NaN,0.60,NaN,Lunch,35,False,NaN,NaN,False
167702,596,2027-05-24 17:30:00,NaN,NaN,0.60,NaN,Dinner,53,False,NaN,NaN,False
167703,596,2027-05-24 20:50:00,NaN,NaN,0.60,NaN,Snack,12,False,NaN,NaN,False


In [ ]:
import os
import math
import pandas as pd
import numpy as np
from pathlib import Path
import xml.dom.minidom 
from datetime import datetime, timedelta
import xml.etree.ElementTree as Et


domtree2018 = xml.dom.minidom.parse("C:/Users/PMI/Documents/TCC_Predicao_glicemia/OhioT1DM/2018/test/559-ws-testing.xml")
domtree2020 = xml.dom.minidom.parse("C:/Users/PMI/Documents/TCC_Predicao_glicemia/OhioT1DM/2020/train/567-ws-training.xml")

# patient2018 = domtree2018.documentElement 
# assert patient2018 is not None
patient2020 = domtree2020.documentElement
assert patient2020 is not None

# print(len(patient2018.getElementsByTagName('sleep')[0].getElementsByTagName('event')))
print(len(patient2020.getElementsByTagName("sleep")[0].getElementsByTagName('event')))


12
